<a href="https://colab.research.google.com/github/SebGalvaleira/mineria-datos-ii-parcial/blob/main/notebooks/e2/E2_99_exportar_evidencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# junta la evidencia de la entrega 2 en un zip con la estructura del repo: evidence/e2/...
import shutil, pandas as pd
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')

D  = Path('/content/drive/MyDrive/cpa_datalake')
EV = Path('/content/evidence/e2')
shutil.rmtree(EV.parent, ignore_errors=True)
for sub in ['logs', 'muestras', 'capturas']:
    (EV / sub).mkdir(parents=True)

# 1) logs tal cual
for f in sorted((D / 'logs').iterdir()):
    if f.is_file():
        shutil.copy(f, EV / 'logs' / f.name)

# 2) inventario: rutas, cantidad de archivos y tamaño por zona y tabla
inv = []
for zona in ['landing', 'bronze', 'silver', 'gold', 'quarantine']:
    for item in sorted((D / zona).iterdir()):
        archivos = [item] if item.is_file() else [p for p in item.rglob('*') if p.is_file()]
        inv.append({'ruta': f'{zona}/{item.name}', 'archivos': len(archivos),
                    'parquet': sum(p.suffix == '.parquet' for p in archivos),
                    'mb': round(sum(p.stat().st_size for p in archivos) / 1e6, 2)})
inventario = pd.DataFrame(inv)
inventario.to_csv(EV / 'inventario_lake.csv', index=False)
display(inventario)

# 3) muestras legibles (CSV) de cada capa
def muestra(ruta, nombre, n=20, orden=None):
    try:
        df = pd.read_parquet(ruta)
        if orden: df = df.sort_values(orden, ascending=False)
        df.head(n).to_csv(EV / 'muestras' / f'{nombre}.csv', index=False)
        print(f'✓ {nombre}: {len(df)} filas en total, {min(n, len(df))} en la muestra')
    except Exception as e:
        print(f'✗ {nombre}: {type(e).__name__} {str(e)[:80]}')

muestra(D / 'silver/usage_events', 'silver_usage_events')
muestra(D / 'gold/org_daily_usage_by_service', 'gold_org_daily_usage_top20', orden='daily_cost_usd')
muestra(D / 'gold/cost_anomaly_mart', 'gold_cost_anomaly_mart_completo', n=10_000)
for q in sorted(p for p in (D / 'quarantine').iterdir() if p.is_dir()):
    if any(q.rglob('*.parquet')):
        muestra(q, f'quarantine_{q.name}')

# 4) zip y descarga
shutil.make_archive('/content/evidence_e2', 'zip', '/content/evidence')
files.download('/content/evidence_e2.zip')

Mounted at /content/drive


,ruta,archivos,parquet,mb
0,landing/billing_monthly.csv,1,0,0.02
1,landing/customers_orgs.csv,1,0,0.01
2,landing/marketing_touches.csv,1,0,0.10
3,landing/nps_surveys.csv,1,0,0.00
4,landing/resources.csv,1,0,0.04
5,landing/support_tickets.csv,1,0,0.07
6,landing/usage_events_stream,120,0,12.93
7,landing/users.csv,1,0,0.07
8,bronze/billing_monthly,6,3,0.02
9,bronze/customers_orgs,2,1,0.01


✓ silver_usage_events: 43200 filas en total, 20 en la muestra
✓ gold_org_daily_usage_top20: 11050 filas en total, 20 en la muestra
✓ gold_cost_anomaly_mart_completo: 289 filas en total, 289 en la muestra
✓ quarantine_muestras_reglas: 4 filas en total, 4 en la muestra


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>